In [17]:
# Imports
import pandas as pd
import duckdb
import numpy as np
import matplotlib.pyplot as plt
import pickle
import glob
from lightgbm import LGBMRegressor
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from sklearn.model_selection import train_test_split

In [18]:
# Use DuckDB to read SQL queries - easiest and most efficient way to attain rows of interest
query = """
    SELECT
        year,
        pitchBucket,
        releaseVelocity,
        inducedVertBreak,
        horzBreak,
        spinRate,
        relX,
        relZ,
        extension,
        phand,
        bhand,
        fb_velo,
        rv
    FROM
        "../../../full_data.parquet"
    WHERE
        year < 2026
"""
df = duckdb.sql(query).df()

In [19]:
# This function intially trains a model on 85% of the data in order to find the best number of boosting rounds
# It then trains the final model on 100% of the data, and saves it as a .pkl file - to be used later
def pred(df, features, target, bucket):

    X = df[features]
    y = df[target]

    # Stage 1: find the right number of trees using a validation split
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.15, random_state=42
    )

    # Initial model, used up to 500 boosting rounds
    probe_model = LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=63,
        min_child_samples=50,
        n_jobs=-1,
        random_state=42
    )

    # Fits initial model with early stopping, to find the optimal amount of boosting rounds
    probe_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric="l2",
        callbacks=[early_stopping(stopping_rounds=50), log_evaluation(period=0)]
    )

    # Defines optimal amount of trees
    best_n_trees = probe_model.best_iteration_
    print(f"best_iteration = {best_n_trees}")

    # Stage 2: refit on 100% of the data using that fixed tree count
    final_model = LGBMRegressor(
        n_estimators=best_n_trees,
        learning_rate=0.05,
        num_leaves=63,
        min_child_samples=50,
        n_jobs=-1,
        random_state=42
    )

    final_model.fit(X, y)

    # Dump final model into pickle file
    with open(f'stuff_{bucket}.pkl', 'wb') as f:
        pickle.dump(final_model, f)

    print(f"Model for {bucket} saved!")

In [20]:
target = ["rv"]

for p in df["pitchBucket"].unique():
    df_tr = df[df["pitchBucket"] == p]
    if p in ["FB", "FC"]:
        features = ["releaseVelocity", "inducedVertBreak", "horzBreak", "spinRate", "relX", "relZ", "extension", "phand", "bhand"]
    else:
        features = ["releaseVelocity", "inducedVertBreak", "horzBreak", "spinRate", "relX", "relZ", "extension", "phand", "bhand", "fb_velo"]
    pred(df_tr, features, target, p)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009693 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1789
[LightGBM] [Info] Number of data points in the train set: 2769032, number of used features: 9
[LightGBM] [Info] Start training from score 0.105771
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[235]	valid_0's l2: 0.13953
best_iteration = 235
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.013101 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1789
[LightGBM] [Info] Number of data points in the train set: 3257685, number of used features: 9
[LightGBM] [Info] Start training from score 0.105728
Model for FB saved!
[LightGBM] [Info] Aut